In [ ]:
import io
import duckdb
from PIL import Image as PILImage
from IPython.display import Video, Image, display
import tempfile
import numpy as np
path = "/home/math4ai/Documents/RCS/robot-control-stack/examples/fr3/digit_2_rs_2_robot"
duckdb.sql(f"describe select *, unnest(obs.frames) from read_parquet('{path}')"), duckdb.sql(f"select count(*) as n_frames from read_parquet('{path}')")

uuids = duckdb.sql(f"SELECT DISTINCT uuid FROM read_parquet('{path}')").fetchnumpy()
episode = 0
n_frames = 3600
frame_stride = 2
image_keys = ["digit_right_left", "digit_right_right"]

uuid1 = uuids["uuid"][episode]
rel = duckdb.read_parquet(path)

print(uuid1)
count = rel.filter(f"uuid='{uuid1}'").count("*").fetchone()[0]
info = rel.filter(f"uuid='{uuid1}'").select("info").fetchone()[0]
success = rel.filter(f"uuid='{uuid1}'").select("success").fetchone()[0]
print(f"episode uuid: {uuid1}, success: {success}, n_steps: {count}, info: {info}")

# Load both cameras in one pass (every frame_stride-th step)
frames = (
    rel.filter(f"uuid='{uuid1}'")
    .filter(f"step % {frame_stride} = 0")
    .select(
        "step, "
        "obs.frames.digit_right_left.rgb.data AS left_data, "
        "obs.frames.digit_right_right.rgb.data AS right_data"
    )
    .order("step")
    .limit(n_frames)
    .fetchall()
)

print(f"Loaded {len(frames)} frames")

def _decode_or_blank(blob, fallback_size=(320, 240)):
    if blob is None:
        return PILImage.new("RGB", fallback_size, color=(0, 0, 0))
    return PILImage.open(io.BytesIO(blob)).convert("RGB")

composite_frames = []
for _, left_data, right_data in frames:
    left = _decode_or_blank(left_data)
    right = _decode_or_blank(right_data, fallback_size=left.size)

    # make same height before concatenation
    if right.size[1] != left.size[1]:
        ratio = left.size[1] / right.size[1]
        right = right.resize((int(right.size[0] * ratio), left.size[1]), resample=PILImage.Resampling.BOX)

    canvas = PILImage.new("RGB", (left.width + right.width, left.height))
    canvas.paste(left, (0, 0))
    canvas.paste(right, (left.width, 0))
    composite_frames.append(canvas)

gif_bytes = make_video_from_frames(
    pil_frames=composite_frames,
    fps=30,
    downscale=0.5,
    format="mp4",   # fast path
)

if gif_bytes[:4] == b"\x00\x00\x00\x20":
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as f:
        f.write(gif_bytes)
        mp4_path = f.name
    display(Video(mp4_path, width=960, embed=True, html_attributes="controls autoplay loop"))
else:
    print("Fallback")
    display(Image(data=gif_bytes, format="gif"))